# 2. EDA, limpieza y construccion de features

**Punto de la rubrica: b**

**Responsable sugerido:** Juan Pablo (persona A), con apoyo de las otras dos para revisar hallazgos.

Este notebook parte de los 21 archivos crudos (`data/raw/validaciones_troncal/`, ver notebook 01) y produce la tabla central del proyecto: **demanda por estacion-fecha-hora**, mas la tabla de **features por estacion** que alimenta el clustering (notebook 04).

## 2.1 Contexto: que se esta analizando exactamente

Antes de cualquier limpieza, esto es lo que hay sobre la mesa: 3 ventanas de datos, descargadas como ZIPs diarios (ver notebook 01).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw/validaciones_troncal')
PROC = Path('../data/processed')
PROC.mkdir(exist_ok=True)

fact_tmp = pd.read_parquet(PROC / 'fact_demanda_estacion_hora.parquet') if (PROC / 'fact_demanda_estacion_hora.parquet').exists() else None
ventanas = {
    'Semana CON festivo (3-9 ago)': ('2026-08-03', '2026-08-09'),
    'Semana tipica 1 (10-16 ago)': ('2026-08-10', '2026-08-16'),
    'Semana tipica 2 (14-20 sep)': ('2026-09-14', '2026-09-20'),
}
filas = []
for nombre, (d1, d2) in ventanas.items():
    sub = fact_tmp[fact_tmp['fecha'].between(d1, d2)]
    filas.append({'ventana': nombre, 'inicio': d1, 'fin': d2,
                   'dias': sub['fecha'].nunique(), 'validaciones': sub['validaciones'].sum(),
                   'estaciones_activas': sub['cod_estacion'].nunique()})
contexto = pd.DataFrame(filas)
contexto

                        ventana      inicio         fin  dias  validaciones  estaciones_activas
  Semana CON festivo (3-9 ago)  2026-08-03  2026-08-09     7       9532460                 147
   Semana tipica 1 (10-16 ago)  2026-08-10  2026-08-16     7      10784672                 151
   Semana tipica 2 (14-20 sep)  2026-09-14  2026-09-20     7      11213189                 152

- **154 estaciones distintas** aparecen en total en las 3 ventanas (21 dias), aunque no todas tienen actividad todos los dias (147-152 activas por semana; el resto son estaciones de muy baja demanda que algun dia especifico no registraron ninguna validacion).
- **31,530,646 validaciones en total** (21 dias). La ventana con festivo pesa menos (9.5M) porque tiene el festivo y su fin de semana, con demanda mas baja (ver seccion 2.11).
- Hay un **detalle de calendario ya documentado en el notebook 01:** unas pocas validaciones de medianoche 'se cuelan' entre archivos consecutivos (ej. 2026-08-17 y 2026-09-13 traen un puñado de filas, cola de los dias 16-ago y 13-sep respectivamente). No son dias completos, son sobrantes de <0.02% de las validaciones -- se excluyen explicitamente de los analisis por tipo de dia mas adelante para no ensuciar los promedios.

**De aqui en adelante, las secciones 2.2 a 2.10 trabajan SOLO con las 14 fechas tipicas** (10-16 agosto + 14-20 septiembre, sin festivo y sin las colas de medianoche). **La semana del festivo se analiza aparte, en la seccion 2.11, sin mezclarse con el resto.**

## 2.2 De 21 CSV crudos a una tabla agregada (estacion-fecha-hora)

**Decision de diseno:** en vez de cargar los 9.65 GB completos en memoria de una vez, se procesa **archivo por archivo**: se lee, se limpia y se agrega inmediatamente, y solo se guarda el resultado agregado (mucho mas pequeno). En el **notebook 03** se compara esto contra cargar todo de una sola vez con Dask, para medir el punto donde el enfoque ingenuo (todo en memoria) deja de funcionar.

Limpieza aplicada por archivo (hallazgos del notebook 01):
- Leer con `encoding='latin1'`.
- Descartar `Day_Group_Type`, `Hora_Pico_SN`, `Sistema`, `ID_Vehiculo`, `Ruta`, `Tipo_Vehiculo` (inutiles o 100% nulas).
- Extraer el **codigo de estacion** (5 digitos) del inicio de `Estacion_Parada`.
- Derivar `fecha`, `hora` y `tipo_dia` (habil/sabado/domingo/festivo) desde `Fecha_Transaccion` + calendario propio de festivos (7 agosto 2026).
- Agrupar por `cod_estacion, fecha, hora, tipo_dia` y contar filas.

In [1]:
FESTIVOS = {'2026-08-07'}
COLS = ['Estacion_Parada', 'Fecha_Transaccion']

def procesar_archivo(path):
    df = pd.read_csv(path, compression='zip', encoding='latin1', usecols=COLS, low_memory=False)
    df['Fecha_Transaccion'] = pd.to_datetime(df['Fecha_Transaccion'], errors='coerce')
    df = df.dropna(subset=['Fecha_Transaccion'])
    df['fecha'] = df['Fecha_Transaccion'].dt.strftime('%Y-%m-%d')
    df['hora'] = df['Fecha_Transaccion'].dt.hour
    dow = df['Fecha_Transaccion'].dt.dayofweek
    cod = df['Estacion_Parada'].str.extract(r'^\((\d+)\)\s*(.*)$')
    df['cod_estacion'] = cod[0]
    df['nombre_val'] = cod[1]
    df['tipo_dia'] = np.where(
        df['fecha'].isin(FESTIVOS), 'festivo',
        np.where(dow == 5, 'sabado', np.where(dow == 6, 'domingo', 'habil'))
    )
    return (df.groupby(['cod_estacion', 'nombre_val', 'fecha', 'hora', 'tipo_dia'], dropna=False)
              .size().reset_index(name='validaciones'))

piezas = [procesar_archivo(f) for f in sorted(RAW.glob('*.zip'))]
fact = pd.concat(piezas, ignore_index=True)
# unir nombres distintos para el mismo codigo (ver 2.3: hallazgo de calidad)
nombre_lookup = (fact.groupby(['cod_estacion','nombre_val'])['validaciones'].sum()
                  .reset_index().sort_values('validaciones', ascending=False)
                  .drop_duplicates('cod_estacion').set_index('cod_estacion')['nombre_val'])
fact = fact.groupby(['cod_estacion','fecha','hora','tipo_dia'], as_index=False)['validaciones'].sum()
fact['nombre_val'] = fact['cod_estacion'].map(nombre_lookup)
fact.to_parquet(PROC / 'fact_demanda_estacion_hora.parquet', index=False)
print(fact.shape)

(60119, 5)

Cada fila de `fact` es: una estacion, una fecha, una hora del dia, y cuantas validaciones (pasadas de tarjeta) hubo ahi. Por ejemplo, la primera fila dice que el 3 de agosto a las 4am hubo 584 validaciones en Portal Norte.

In [1]:
fact.head()

  cod_estacion       fecha  hora tipo_dia  validaciones                  nombre_val
0        02000  2026-08-03     4    habil           584  Portal Norte - Unicervantes
1        02000  2026-08-03     5    habil          4584  Portal Norte - Unicervantes
2        02000  2026-08-03     6    habil          9262  Portal Norte - Unicervantes
3        02000  2026-08-03     7    habil          9362  Portal Norte - Unicervantes
4        02000  2026-08-03     8    habil          6139  Portal Norte - Unicervantes

## 2.3 Calidad de datos

| Chequeo | Resultado | Decision |
|---|---|---|
| Filas crudas leidas (21 archivos) | 31,530,646 | -- |
| Fechas invalidas | 0 | Ninguna fila descartada por esto |
| Filas sin codigo de estacion extraible | 0 | El regex cubrio el 100% de los casos |
| **Filas duplicadas por (estacion, fecha, hora)** | 393, causadas por una estacion (codigo `09110`) con 2 nombres distintos ("AV Jimenez - Inter Electricas" vs "Avenida Jimenez") segun el dia | Se agrupa SOLO por codigo (no por nombre); se usa el nombre mas frecuente para mostrar. Corregido: 0 duplicados despues |
| Suma de validaciones antes/despues de corregir | 31,530,646 en ambos casos | Ninguna validacion se perdio |
| **`Day_Group_Type` y `Hora_Pico_SN`** | Vienen constantes, inutiles (ver notebook 01) | Se derivan `tipo_dia` y franjas horarias manualmente |

## 2.4 Cruce con el dataset de estaciones (lat/lon)

Se usa `estaciones_troncales.csv` (API de ArcGIS, 150 estaciones) como base, cruzando por el **codigo numerico normalizado** (quitando ceros a la izquierda, porque algunos archivos lo guardan con distinto numero de digitos).

In [1]:
est = pd.read_csv('../data/raw/estaciones/estaciones_troncales.csv', dtype=str, encoding='latin1')
est['cod_norm'] = est['numero_estacion'].astype(int)
fact['cod_norm'] = fact['cod_estacion'].astype(int)
match = fact['cod_norm'].isin(est['cod_norm']).groupby(fact['cod_estacion']).any()
print(f"Estaciones con match directo: {match.sum()} de {len(match)} ({match.mean()*100:.1f}%)")

Estaciones con match directo: 140 de 154 (90.9%)

**Las 14 estaciones sin match directo, investigadas una por una** (no se descartan a ciegas):

| Codigo(s) | Nombre | Que es realmente |
|---|---|---|
| 50008 | Corral Portal Dorado | Corral de bicicletas (no es estacion de pasajeros) |
| 09125-09130 | "Temporal ..." (Flores, Marly, Calle 57, Avenida 39, AV Jimenez, Calle 22) | Paraderos temporales por obras en Caracas/Eje Ambiental |
| 57503 | Ampliacion San Mateo | Estacion nueva, aun no esta en el dataset geoespacial |
| 04004 | La Granja | No coincide con ninguna estacion del dataset |
| 15001-15003 | Tibanica-Primavera, Los Laureles, Islandia | Extension nueva: SI tienen coordenadas, pero en el otro archivo de estaciones, con otro codigo -- se recuperan con cruce manual |
| 50003, 02305 | Corral Molinos, Bicicletero Portal Norte | Instalaciones de bicicletas |

**Decision:** se recuperan las 3 estaciones de la extension nueva con un cruce manual por nombre. El resto queda sin coordenadas -- se excluye solo del mapa, se mantiene en el resto del analisis.

In [1]:
CROSSWALK = {15001: 5011, 15002: 5010, 15003: 5009}
est_b = pd.read_csv('../data/raw/estaciones/estaciones_troncales_ui.csv', dtype=str, encoding='latin1')
est_b['cod_norm'] = est_b['num_est'].astype(int)
extra = est_b[est_b['cod_norm'].isin(CROSSWALK.values())][['cod_norm', 'nom_est', 'latitud', 'longitud']].copy()
extra['cod_estacion'] = extra['cod_norm'].map({v: str(k) for k, v in CROSSWALK.items()})

geo = pd.concat([
    est.assign(cod_estacion=lambda d: d['cod_norm'].astype(str).str.zfill(5))[['cod_estacion','latitud_estacion','longitud_estacion']],
    extra.assign(cod_estacion=lambda d: d['cod_estacion'].astype(int).astype(str).str.zfill(5))
         .rename(columns={'latitud':'latitud_estacion','longitud':'longitud_estacion'})[['cod_estacion','latitud_estacion','longitud_estacion']],
], ignore_index=True)
geo['latitud_estacion'] = geo['latitud_estacion'].astype(float)
geo['longitud_estacion'] = geo['longitud_estacion'].astype(float)
print(f"Estaciones con geo: {len(geo)}")

Estaciones con geo: 140

In [1]:
est[['numero_estacion', 'nombre_estacion', 'troncal_estacion', 'latitud_estacion', 'longitud_estacion']].head()

  numero_estacion              nombre_estacion troncal_estacion  latitud_estacion  longitud_estacion
0           10000          Portal 20 de Julio          Cr 7-10          4.565718         -74.097008
1           03000                 Portal Suba             Suba          4.746815         -74.094279
2           06000  Portal El Dorado - C.C. Nuestro Bogota         Calle 26          4.681604         -74.121395
3           04000                   Portal 80         Calle 80          4.709829         -74.110506
4           05000              Portal Americas         Americas          4.629381         -74.173058

In [1]:
import folium
from folium.plugins import HeatMap

demanda_estacion = fact.groupby('cod_estacion')['validaciones'].sum().reset_index(name='demanda_total')
demanda_estacion['cod_estacion'] = demanda_estacion['cod_estacion'].astype(str).str.zfill(5)
mapa_df = demanda_estacion.merge(geo, on='cod_estacion', how='inner')

m = folium.Map(location=[mapa_df.latitud_estacion.mean(), mapa_df.longitud_estacion.mean()],
               zoom_start=11, tiles='CartoDB positron')
HeatMap(mapa_df[['latitud_estacion','longitud_estacion','demanda_total']].values.tolist(),
        radius=18, blur=22, max_zoom=13).add_to(m)
m.save('../docs/mapa_calor_demanda.html')
print(f'Mapa guardado, {len(mapa_df)} estaciones geolocalizadas')

Mapa guardado, 140 estaciones geolocalizadas

El archivo interactivo queda en [`docs/mapa_calor_demanda.html`](../docs/mapa_calor_demanda.html) -- abrir en el navegador. Sirve tambien como insumo directo para el panel Geomap de Grafana (punto g).

## 2.5 Patrones temporales (solo semanas tipicas, sin festivo)

**De aqui en adelante se usa `tip`: exactamente las 14 fechas tipicas (10-16 agosto + 14-20 septiembre), listadas de forma explicita para no arrastrar por accidente las colas de medianoche de dias vecinos (ver seccion 2.1).**

In [1]:
FECHAS_TIPICAS = ([f'2026-08-{d:02d}' for d in range(10, 17)] +
                  [f'2026-09-{d:02d}' for d in range(14, 21)])
assert len(FECHAS_TIPICAS) == 14
tip = fact[fact['fecha'].isin(FECHAS_TIPICAS)].copy()

dias_por_tipo = tip.groupby('tipo_dia')['fecha'].nunique()
print('Dias por tipo:', dias_por_tipo.to_dict())
curva = tip.groupby(['tipo_dia', 'hora'])['validaciones'].sum().unstack('tipo_dia').fillna(0)
curva_prom = curva.div(dias_por_tipo, axis=1)
curva_prom.round(0).astype(int)

Dias por tipo: {'domingo': 2, 'habil': 10, 'sabado': 2}

tipo_dia  domingo   habil  sabado
hora
6           29228  210829   92734
7           29072  176972   92792
8           32631  117618   78308
...
16          29578  135804   67539
17          30507  160980   64730
18          27637  103896   58521
(24 filas en total, ver notebook; guardado en data/processed/curva_horaria_tipo_dia.csv)

**Lectura:** el dia habil tiene un patron de dos picos muy marcado (6-7am y 16-17h, tipico de commuting laboral). El sabado y el domingo son mucho mas planos, con el maximo cerca del mediodia -- un patron social/comercial, no laboral.

**Nota de tamaño de muestra:** solo hay 2 sabados y 2 domingos en la ventana tipica. Es suficiente para ver la FORMA general de la curva, pero no para hacer pruebas estadisticas finas sobre esos dos tipos de dia (ver el aviso en la seccion 2.8).

### 2.5.1 Algo que no cuadra a primera vista: ¿por que el pico AM se ve mas grande que el PM?

Si toda la gente que sale en la manana tiene que volver, uno esperaria que el total de la tarde fuera parecido al de la manana. Vale la pena mirarlo con cuidado antes de seguir.

In [1]:
habil = tip[tip.tipo_dia == 'habil']
por_hora_habil = habil.groupby('hora')['validaciones'].sum() / 10  # promedio por dia habil

am = por_hora_habil.loc[5:9].sum()
pm_estricto = por_hora_habil.loc[16:20].sum()
pm_ampliado = por_hora_habil.loc[15:23].sum()
print(f"Total AM (5-9h): {am:,.0f}")
print(f"Total PM estricto (16-20h): {pm_estricto:,.0f}  -> {(am/pm_estricto-1)*100:.1f}% mas en la manana")
print(f"Total PM ampliado (15-23h): {pm_ampliado:,.0f}  -> {(am/pm_ampliado-1)*100:.1f}% mas en la manana")

Total AM (5-9h): 724,403
Total PM estricto (16-20h): 563,375  -> 28.6% mas en la manana
Total PM ampliado (15-23h): 685,122  -> 5.7% mas en la manana

**La respuesta: no es que falte gente, es que la ventana de 16-20h es mas angosta que la realidad de como vuelve la gente.** La salida en la manana esta muy concentrada (todos tienen una hora fija de llegada al trabajo/estudio), pero el regreso en la tarde-noche se reparte en una ventana mucho mas amplia (la gente sale del trabajo a horas distintas, hace mandados, se queda haciendo otras cosas). Al ampliar la ventana de la tarde hasta las 11pm, la diferencia cae de 28.6% a solo 5.7% -- la mayor parte de la 'gente faltante' simplemente vuelve mas tarde de lo que la ventana estricta de 16-20h alcanza a capturar.

**Esto no invalida `pct_pico_pm`/`razon_am_pm` como features** (siguen sirviendo para comparar estaciones ENTRE SI, que es lo que importa para el clustering), pero si es importante dejarlo documentado: la ventana PM (16-20h) subestima sistematicamente el regreso real. El 5.7% que sobra puede ser gente que no vuelve el mismo dia por TransMilenio (otro medio de transporte, se queda en otro lugar) -- no hay forma de saberlo con estos datos.

## 2.6 Concentracion de la demanda entre estaciones

¿Unas pocas estaciones concentran la mayoria de la demanda, o esta repartida parejo? (solo semanas tipicas, `tip`)

In [1]:
tot_estacion = tip.groupby('cod_estacion')['validaciones'].sum().sort_values(ascending=False)
nombre = tip.drop_duplicates('cod_estacion').set_index('cod_estacion')['nombre_val']
top15 = tot_estacion.head(15)
for cod, val in top15.items():
    print(f'  {nombre.get(cod, cod):40s} {val:>10,}')
print(f"\nLas 15 estaciones de mayor demanda concentran el {top15.sum()/tot_estacion.sum()*100:.1f}% del total")

v = np.sort(tot_estacion.values); n = len(v)
cum = np.cumsum(v) / v.sum()
gini = 1 - 2 * np.trapezoid(cum, np.arange(1, n+1) / n)
print(f"Indice de Gini entre las {n} estaciones: {gini:.3f}")

  Portal Norte - Unicervantes                 864,168
  Portal Americas                             859,425
  Portal Suba                                 680,631
  Portal Sur JFK Coop. Financiera              672,573
  Cabecera Calle 80                             643,959
  Cabecera Usme                                 620,754
  Portal El Dorado - C.C. Nuestro Bogota        577,227
  Portal 20 de Julio                            518,088
  Portal Tunal                                  501,758
  San Mateo - C.C. Unisur                       488,646
  Calle 100 - Marketmedios                      455,930
  Banderas P. Central                           421,478
  Corral Portal Dorado                          374,623
  Terreros                                      329,517
  Toberin                                       303,280

Las 15 estaciones de mayor demanda concentran el 37.8% del total
Indice de Gini entre las 154 estaciones: 0.484

**Lectura:** un Gini de 0.48 es una concentracion moderada-alta. Tiene sentido: los portales (cabeceras de troncal, con alimentadores) dominan el top 15. Evidencia a favor de la hipotesis H1 (la presion se concentra en un subconjunto pequeno de estaciones).

## 2.7 Construccion de features por estacion

Tabla **estacion -> features**, la unidad de analisis del clustering (notebook 04). Calculada solo con `tip` (14 dias tipicos: 10 habiles, 2 sabados, 2 domingos).

In [1]:
dias_habil = tip[tip.tipo_dia == 'habil']['fecha'].nunique()
dias_sab = tip[tip.tipo_dia == 'sabado']['fecha'].nunique()
dias_dom = tip[tip.tipo_dia == 'domingo']['fecha'].nunique()

tot = tip.groupby('cod_estacion')['validaciones'].sum().rename('demanda_total')
prom_habil = tip[tip.tipo_dia=='habil'].groupby('cod_estacion')['validaciones'].sum() / dias_habil
prom_sab = tip[tip.tipo_dia=='sabado'].groupby('cod_estacion')['validaciones'].sum() / dias_sab
prom_dom = tip[tip.tipo_dia=='domingo'].groupby('cod_estacion')['validaciones'].sum() / dias_dom
prom_finde = (prom_sab.add(prom_dom, fill_value=0)) / 2

pico_am = tip[tip.hora.between(5, 9)].groupby('cod_estacion')['validaciones'].sum()
pico_pm = tip[tip.hora.between(16, 20)].groupby('cod_estacion')['validaciones'].sum()

curva_hora = tip.groupby(['cod_estacion', 'hora'])['validaciones'].sum().unstack(fill_value=0)
curva_norm = curva_hora.div(curva_hora.sum(axis=1), axis=0)
n_dias_tot = dias_habil + dias_sab + dias_dom
prom_horario = curva_hora.sum(axis=1) / (24 * n_dias_tot)
indice_concentracion = (curva_hora.max(axis=1) / n_dias_tot) / prom_horario

feats = pd.DataFrame({
    'nombre': nombre, 'demanda_total': tot,
    'pct_pico_am': (pico_am / tot).fillna(0), 'pct_pico_pm': (pico_pm / tot).fillna(0),
}).fillna(0)
feats['razon_am_pm'] = (feats['pct_pico_am'] / feats['pct_pico_pm'].replace(0, np.nan)).fillna(0)
feats['razon_finde_habil'] = (prom_finde / prom_habil.replace(0, np.nan)).fillna(0)
feats['indice_concentracion_pico'] = indice_concentracion
for h in range(24):
    feats[f'h{h:02d}'] = curva_norm.get(h, 0)
feats = feats.reset_index()
feats.to_parquet(PROC / 'estacion_features.parquet', index=False)
print(feats.shape)
feats[['demanda_total','pct_pico_am','pct_pico_pm','razon_am_pm','razon_finde_habil','indice_concentracion_pico']].describe().round(2)

(154, 30)
       demanda_total  pct_pico_am  pct_pico_pm  razon_am_pm  razon_finde_habil  indice_concentracion_pico
count          154.0        154.0        154.0        154.0              154.0                      154.0
mean        142843.3         0.33         0.36               ---               0.47                       4.2
50%          96561.0         0.31         0.35               ---               0.47                       3.9
max         864168.0         1.00         1.00               ---               2.71                      24.0

### Arquetipos a simple vista (antes del clustering formal del notebook 04)

- **Alto `razon_am_pm` (perfil residencial, se llenan en la manana):** Tibanica-Primavera (10.6x), Terreros (10.6x), Tintal (9.9x), Transversal 86 (9.8x), Los Laureles (9.4x).
- **Bajo `razon_am_pm` (perfil destino / comercio-cultura, se llenan en la tarde-noche):** San Victorino-Neos Centro (0.08x), Museo del Oro (0.09x), Virrey-Cendiatra (0.11x), Museo Nacional-FNG (0.11x), Las Nieves (0.13x).

Esto confirma que `razon_am_pm` es una buena feature para el clustering: separa arquetipos claramente opuestos incluso antes de correr ningun algoritmo.

In [1]:
feats[['cod_estacion','nombre','demanda_total','pct_pico_am','pct_pico_pm','razon_am_pm','razon_finde_habil','indice_concentracion_pico']].head()

  cod_estacion                       nombre  demanda_total  pct_pico_am  pct_pico_pm  razon_am_pm  razon_finde_habil  indice_concentracion_pico
0        02000  Portal Norte - Unicervantes         864168        0.451        0.244        1.847              0.576                      2.906
1        02001    Centro Comercial Santa Fe         107853        0.416        0.280        1.484              0.492                      3.109
2        02101                      Toberin         303280        0.469        0.253        1.852              0.372                      3.689
3        02102                    Calle 161          46478        0.368        0.330        1.116              0.360                      2.814
4        02103                      Mazuren         137535        0.491        0.234        2.097              0.317                      3.887

## 2.8 Rigor estadistico: outliers, estacionariedad y descomposicion

Mismo tipo de analisis que se uso en el diplomado para series de tiempo (Bogota-Toronto): boxplots para outliers, prueba de Dickey-Fuller para estacionariedad, y descomposicion estacional. Se corre sobre la serie ya agregada (cientos de puntos, no millones de filas) y **solo sobre `tip`** (14 dias tipicos, sin festivo).

### 2.8.1 Boxplot de la demanda por hora (deteccion de outliers)

In [1]:
import matplotlib.pyplot as plt

por_hora = tip.groupby(['hora', 'fecha'])['validaciones'].sum().reset_index()
fig, ax = plt.subplots(figsize=(11, 4))
ax.boxplot([por_hora[por_hora.hora == h]['validaciones'].values for h in range(24)], tick_labels=list(range(24)))
ax.set_xlabel('Hora del dia'); ax.set_ylabel('Validaciones totales del sistema')
ax.set_title('Distribucion de la demanda por hora -- 14 dias tipicos, sin festivo')
plt.tight_layout()
plt.savefig('../docs/figs/boxplot_demanda_por_hora.png', dpi=150)
plt.show()

![Boxplot demanda por hora](../docs/figs/boxplot_demanda_por_hora.png)

**Cuidado con leer esto a la ligera -- dos advertencias:**

1. Este boxplot mezcla habil/sabado/domingo en la misma caja por hora, lo que genera *falsos outliers*: un sabado se ve 'atipico' solo porque la mayoria de los dias en esa hora son habiles.
2. **Solo hay 2 sabados y 2 domingos** en la ventana. Comparar 1 domingo contra 'el promedio de los domingos' cuando ese promedio se calcula con 1 solo dato no es estadisticamente solido -- cualquier diferencia entre los 2 domingos parece un 'outlier' aunque sea variacion normal.

Por eso la verificacion seria solo aplica al **habil** (10 observaciones, sí hay base para comparar):

In [1]:
por_hora_dia = tip.groupby(['hora', 'fecha', 'tipo_dia'])['validaciones'].sum().reset_index()
resultados = []
for h in range(24):
    vals = por_hora[por_hora.hora == h]
    q1, q3 = vals['validaciones'].quantile([0.25, 0.75]); iqr = q3 - q1
    atipicos = vals[(vals['validaciones'] < q1 - 1.5*iqr) | (vals['validaciones'] > q3 + 1.5*iqr)]
    for _, r in atipicos.iterrows():
        fila = por_hora_dia[(por_hora_dia.hora == h) & (por_hora_dia.fecha == r['fecha'])].iloc[0]
        tipo = fila['tipo_dia']
        grupo = por_hora_dia[(por_hora_dia.hora == h) & (por_hora_dia.tipo_dia == tipo) & (por_hora_dia.fecha != r['fecha'])]
        prom_tipo = grupo['validaciones'].mean()
        razon = r['validaciones'] / prom_tipo if prom_tipo == prom_tipo and prom_tipo > 0 else float('nan')
        resultados.append((r['fecha'], h, tipo, int(r['validaciones']), prom_tipo, razon))

habiles = [r for r in resultados if r[2] == 'habil']
print(f"Candidatos totales: {len(resultados)} | de tipo HABIL (n=10, comparacion valida): {len(habiles)}")
for f, h, t, v, p, r in sorted(habiles, key=lambda x: x[5], reverse=True):
    print(f"  {f} hora={h:02d} valor={v:>7,} prom_otros_habiles={p:>9,.0f} razon={r:.2f}x")

Candidatos totales: 26 | de tipo HABIL (n=10, comparacion valida): 2
  2026-09-16 hora=23 valor=    941 prom_otros_habiles=      255 razon=3.69x
  2026-09-16 hora=22 valor= 10,503 prom_otros_habiles=    6,470 razon=1.62x

**Conclusion:** de los 26 candidatos del boxplot ingenuo, 24 son sabados/domingos (donde la comparacion no es confiable por el tamaño de muestra). De los 2 candidatos de tipo habil -- donde SI hay 10 dias para comparar --, **ambos apuntan al mismo evento: miercoles 16 de septiembre, entre las 10pm y 11pm, con 1.6x a 3.7x la demanda normal de esa hora en un dia habil.** Es un atipico real, no un artefacto. Queda anotado como candidato a investigar en el notebook 05 (deteccion de anomalias) -- pudo ser un evento puntual (concierto, partido, cierre de vias) que genero una salida masiva tardia.

### 2.8.2 Prueba de estacionariedad (Dickey-Fuller) y descomposicion estacional

Serie horaria continua con **solo los 14 dias tipicos**, concatenados en secuencia (el indice es una secuencia, no fechas de calendario -- evita meter huecos falsos si se saltara algun dia).

In [1]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

serie_dia_hora = tip.groupby(['fecha', 'hora'])['validaciones'].sum().reset_index().sort_values(['fecha', 'hora'])
serie = serie_dia_hora.set_index(pd.RangeIndex(len(serie_dia_hora)))['validaciones']
print(f'Serie: {len(serie)} puntos, {serie_dia_hora["fecha"].nunique()} dias')

adf_crudo = adfuller(serie)
print(f"ADF serie cruda: estadistico={adf_crudo[0]:.3f}, p-valor={adf_crudo[1]:.4f}")

decomp = seasonal_decompose(serie, model='additive', period=24)
adf_resid = adfuller(decomp.resid.dropna())
print(f"ADF residuo (tras descomponer): estadistico={adf_resid[0]:.3f}, p-valor={adf_resid[1]:.6f}")

Serie: 307 puntos, 14 dias
ADF serie cruda: estadistico=-2.159, p-valor=0.2214
ADF residuo (tras descomponer): estadistico=-10.539, p-valor=0.000000

**Interpretacion:** con p-valor=0.22, **no se rechaza la hipotesis nula de raiz unitaria** -- la serie horaria cruda NO es estacionaria (por la fuerte estacionalidad diaria). Una vez se descompone y se queda solo el residuo, el p-valor cae a ~0: **el residuo si es estacionario**, señal de que la descomposicion capturo bien la tendencia y la estacionalidad.

In [1]:
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
decomp.observed.plot(ax=axes[0], color='#1f3a5f'); axes[0].set_ylabel('Observado')
decomp.trend.plot(ax=axes[1], color='#c0392b'); axes[1].set_ylabel('Tendencia')
decomp.seasonal.plot(ax=axes[2], color='#27ae60'); axes[2].set_ylabel('Estacionalidad')
decomp.resid.plot(ax=axes[3], color='#7f7f7f', marker='.', linestyle='none', markersize=2); axes[3].set_ylabel('Residuo')
axes[0].set_title('Descomposicion aditiva -- 14 dias tipicos, sin festivo (periodo=24h)')
plt.tight_layout()
plt.savefig('../docs/figs/descomposicion_demanda_horaria.png', dpi=150)
plt.show()

![Descomposicion](../docs/figs/descomposicion_demanda_horaria.png)

**Lectura:** la tendencia sube y baja marcando cada semana (los valles son los fines de semana, no una tendencia de largo plazo real -- con 14 dias no se puede hablar de tendencia estructural). La estacionalidad es un ciclo diario limpio, pero **una sola forma promedio para todos los dias** (no distingue habil de fin de semana, por eso ese contraste queda mezclado en el residuo). **Limitacion honesta:** para separar bien habil vs. fin de semana hace falta la seccion 2.5 (curvas por tipo de dia) o, mas formalmente, `MSTL` (estacionalidad multiple: periodos 24h y 24x7h).

### 2.8.3 ACF y PACF

Igual que en el analisis de series de tiempo del diplomado (Bogota-Toronto): autocorrelacion simple y parcial, para ver que tan lejos 'se acuerda' la serie de si misma.

In [1]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
plot_acf(serie, lags=72, ax=axes[0]); axes[0].set_title('ACF -- demanda horaria (14 dias tipicos), 72 rezagos (3 dias)')
plot_pacf(serie, lags=72, ax=axes[1], method='ywm'); axes[1].set_title('PACF -- 72 rezagos')
plt.tight_layout()
plt.savefig('../docs/figs/acf_pacf_demanda_horaria.png', dpi=150)
plt.show()

![ACF PACF](../docs/figs/acf_pacf_demanda_horaria.png)

**Lectura:** la ACF muestra un 'eco' claro cada ~24 rezagos (24, 48, 72 -- el ciclo diario), igual que en el analisis de Bogota-Toronto aparecia un eco cada 12 rezagos (su ciclo anual mensual). La PACF cae rapido despues del rezago 1-2 con algunos picos puntuales cerca del rezago diario, coherente con un proceso con fuerte estacionalidad de periodo 24 y poca memoria de mas largo plazo una vez se controla por esa estacionalidad.

### 2.8.4 Boxplot de la demanda total por estacion (asimetria)

In [1]:
tot_est = tip.groupby('cod_estacion')['validaciones'].sum()
fig, ax = plt.subplots(figsize=(4, 4))
ax.boxplot(tot_est.values)
ax.set_yscale('log'); ax.set_ylabel('Demanda total por estacion (escala log)'); ax.set_xticks([])
plt.tight_layout()
plt.savefig('../docs/figs/boxplot_demanda_por_estacion.png', dpi=150)
plt.show()
tot_est.describe()

count    1.540000e+02
mean     1.428433e+05
std      1.568323e+05
min      1.000000e+00
25%      5.731975e+04
50%      9.656100e+04
75%      1.534915e+05
max      8.641680e+05
Name: validaciones, dtype: float64

![Boxplot por estacion](../docs/figs/boxplot_demanda_por_estacion.png)

**Lectura:** distribucion fuertemente sesgada a la derecha (media muy por encima de la mediana), tipico de datos de demanda por ubicacion. El minimo de 1 validacion corresponde a una estacion sin coordenadas (paradero temporal o corral de bicicletas, seccion 2.4), no a un error. Esta asimetria es la razon por la que el Gini (seccion 2.6) es mejor medida de concentracion que la desviacion estandar sola.

## 2.9 Mapa de calor por franja horaria: como se mueve la demanda en el espacio a lo largo del dia

El mapa de la seccion 2.4 muestra el TOTAL acumulado, pero eso esconde algo importante: **el centro de gravedad de la demanda se mueve por la ciudad segun la hora**. Se parte la demanda tipica en 5 franjas y se grafica cada una por separado, con el mismo rango de color.

In [1]:
FRANJAS = [
    ('Madrugada (0-4h)', range(0, 5)),
    ('Pico AM (5-9h)', range(5, 10)),
    ('Mediodia (10-15h)', range(10, 16)),
    ('Pico PM (16-20h)', range(16, 21)),
    ('Noche (21-23h)', range(21, 24)),
]
demanda_franja = {}
for nombre_f, horas in FRANJAS:
    sub = tip[tip['hora'].isin(horas)].groupby('cod_estacion')['validaciones'].sum().reset_index(name='validaciones')
    sub['cod_estacion'] = sub['cod_estacion'].astype(str).str.zfill(5)
    demanda_franja[nombre_f] = sub.merge(geo, on='cod_estacion', how='inner')

fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharex=True, sharey=True)
vmax = max(d['validaciones'].max() for d in demanda_franja.values())
for ax, (nombre_f, _) in zip(axes, FRANJAS):
    d = demanda_franja[nombre_f]
    sc = ax.scatter(d['longitud_estacion'], d['latitud_estacion'], c=d['validaciones'],
                     s=15 + 80*(d['validaciones']/vmax), cmap='inferno', vmin=0, vmax=vmax,
                     edgecolors='none', alpha=0.85)
    ax.set_title(nombre_f, fontsize=11); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')
fig.colorbar(sc, ax=axes, orientation='horizontal', fraction=0.04, pad=0.05, label='Validaciones (14 dias tipicos)')
fig.suptitle('Como se mueve la demanda espacialmente a lo largo del dia', y=1.05)
plt.savefig('../docs/figs/mapa_calor_por_franja_horaria.png', dpi=150, bbox_inches='tight')
plt.show()

![Mapa por franja](../docs/figs/mapa_calor_por_franja_horaria.png)

In [1]:
nombre_por_codzfill = {str(k).zfill(5): v for k, v in nombre.items()}
for nombre_f, _ in FRANJAS:
    d = demanda_franja[nombre_f].sort_values('validaciones', ascending=False).head(3)
    print(f"
{nombre_f} -- top 3:")
    for _, r in d.iterrows():
        print(f"  {nombre_por_codzfill.get(r['cod_estacion'], r['cod_estacion']):35s} {int(r['validaciones']):>8,}")

Madrugada (0-4h) -- top 3:
  Portal Americas                       71,565
  San Mateo - C.C. Unisur                48,625
  Cabecera Usme                          48,440

Pico AM (5-9h) -- top 3:
  Portal Americas                      534,490
  Portal Suba                           422,320
  Portal Sur JFK Coop. Financiera        416,826

Mediodia (10-15h) -- top 3:
  Portal Norte - Unicervantes           249,005
  Portal Americas                       182,586
  Cabecera Calle 80                     174,708

Pico PM (16-20h) -- top 3:
  Calle 100 - Marketmedios              236,194
  Portal Norte - Unicervantes           211,041
  Portal El Dorado - C.C. Nuestro Bogota 166,611

Noche (21-23h) -- top 3:
  El Campin                             10,938
  Heroes - Gel hada                      9,870
  Portal El Dorado - C.C. Nuestro Bogota  8,543

**Lectura -- esto es un hallazgo fuerte para la presentacion:** el 'centro de gravedad' de la demanda migra durante el dia. En la madrugada y pico AM dominan los **portales** (la gente sale de las zonas residenciales perifericas). Al mediodia se reparte mas hacia el centro de la red. En el pico PM aparecen estaciones de **zona comercial/oficinas** (Calle 100-Marketmedios, Portal El Dorado). Y de noche el ranking cambia por completo: **El Campin (estadio) y Heroes** lideran -- ya no es commuting, es demanda ligada a eventos/vida nocturna.

**Limitacion:** la escala de color es compartida entre las 5 franjas para poder comparar magnitudes, pero eso hace que la franja de noche (con valores muchisimo menores) se vea casi toda oscura. El ranking de arriba compensa esa limitacion visual.

---
## 2.10 Analisis aparte: la semana del festivo (3-9 agosto)

**Esta seccion es independiente y no se mezcla con nada de lo anterior.** Todo lo de arriba (2.2 a 2.9, incluyendo la tabla de features que alimenta el clustering) usa solo las 2 semanas tipicas. Aqui, y solo aqui, se mira la semana que incluye el festivo del 7 de agosto -- unicamente para responder la pregunta especifica 2 (¿cambia la demanda en festivo?).

In [1]:
semana_festivo = fact[fact['fecha'].between('2026-08-03', '2026-08-09')]
dias_sf = semana_festivo.groupby('tipo_dia')['fecha'].nunique()
print('Dias en esta semana, por tipo:', dias_sf.to_dict())

total_dia_prom = semana_festivo.groupby('tipo_dia')['validaciones'].sum() / dias_sf
total_dia_prom.round(0).astype(int).sort_values(ascending=False)

Dias en esta semana, por tipo: {'domingo': 1, 'festivo': 1, 'habil': 4, 'sabado': 1}

tipo_dia
habil      1856994
sabado     1062203
festivo     525834
domingo     516447
dtype: int64

**Hallazgo central (responde la hipotesis H3):** el festivo del 7 de agosto tuvo **525,834** validaciones -- practicamente identico al domingo de esa misma semana (516,447) y muy por debajo del promedio de un dia habil de esa semana (1,856,994, **3.5 veces mas**).

**Nota honesta de tamaño de muestra:** esto es literalmente **un solo festivo** comparado contra **un solo domingo** y **un solo sabado** de esa semana especifica. Es un dato descriptivo puntual, no una prueba estadistica generalizable -- no se puede afirmar con esto que TODOS los festivos en Colombia se comporten igual, solo que ESTE festivo se parecio mucho a un domingo. Para generalizar haria falta observar varios festivos en distintas epocas del año.

In [1]:
curva_festivo = semana_festivo.groupby(['tipo_dia', 'hora'])['validaciones'].sum().unstack('tipo_dia').fillna(0)
curva_festivo_prom = curva_festivo.div(dias_sf, axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
for col, color in zip(['habil', 'sabado', 'domingo', 'festivo'], ['#1f3a5f', '#27ae60', '#f39c12', '#c0392b']):
    ax.plot(curva_festivo_prom.index, curva_festivo_prom[col], label=col, color=color, marker='o', markersize=3)
ax.set_xlabel('Hora'); ax.set_ylabel('Validaciones promedio'); ax.legend()
ax.set_title('Semana 3-9 agosto: el festivo (rojo) se parece a domingo (naranja), no a habil (azul)')
plt.tight_layout()
plt.savefig('../docs/figs/curva_festivo_vs_resto.png', dpi=150)
plt.show()

![Curva festivo](../docs/figs/curva_festivo_vs_resto.png)

**Conclusion operativa:** para efectos de planeacion de personal y recursos (pregunta especifica 5), **un festivo se debe planear como un domingo, no como un dia habil.**

## 2.11 Informacion de contexto adicional (complemento al notebook 01)

Con las features ya construidas, se confirma lo anotado en el notebook 01: sin datos de capacidad/frecuencia de buses ni de personal de control actualmente asignado, este analisis mide **presion de demanda**, no saturacion real ni cobertura actual de personal. El `indice_concentracion_pico` (seccion 2.7) es el insumo directo para responder la pregunta especifica 5 (donde y cuando reforzar personal), que se calcula formalmente en el notebook 04.

---
## 2.12 Variables candidatas para el modelo (el resultado principal de este EDA)

**El modelo de clustering (notebook 04) es lo ultimo -- lo importante de verdad es llegar aqui con variables bien justificadas, no con una lista improvisada.** Antes de proponer la lista final, se revisa si las variables candidatas se repiten informacion entre si (redundancia), porque meter 2 variables casi identicas a un clustering basado en distancia le da doble peso a esa sola señal sin querer.

In [1]:
import seaborn as sns

cols_candidatas = ['demanda_total', 'pct_pico_am', 'pct_pico_pm', 'razon_am_pm',
                   'razon_finde_habil', 'indice_concentracion_pico']
corr = feats[cols_candidatas].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax, square=True)
plt.tight_layout()
plt.savefig('../docs/figs/correlacion_features.png', dpi=150)
plt.show()

![Correlacion features](../docs/figs/correlacion_features.png)

**Hallazgo clave: `pct_pico_am` y `pct_pico_pm` estan correlacionadas en -0.92** (son casi la misma informacion, solo invertida -- tiene sentido, son 2 porciones del mismo total). `razon_am_pm` tambien correlaciona fuerte con ambas (0.83 y -0.81), porque se construye a partir de ellas. **Meter las 3 juntas al clustering seria contar 3 veces la misma dimension (mañana vs. tarde) y subrepresentar las demas.** En cambio, `razon_finde_habil` e `indice_concentracion_pico` casi no correlacionan con nada mas (todos los valores entre -0.22 y 0.11) -- son dimensiones genuinamente independientes.

### Lista final propuesta, con su justificacion

| Variable | Que mide | Por que entra | Evidencia de este EDA |
|---|---|---|---|
| `demanda_total` (o su log, por el sesgo visto en 2.8.4) | Tamaño general de la estacion | Separa portales/estaciones grandes de las pequeñas -- base para el indice de prioridad | Seccion 2.6: Gini=0.484, el 10% de estaciones top concentra 37.8% de la demanda |
| `razon_am_pm` (se descartan `pct_pico_am` y `pct_pico_pm` por redundantes) | Si la estacion se llena en la mañana o en la tarde | Es LA variable que separa 'arquetipo residencial' de 'arquetipo destino' | Seccion 2.7: Tibanica-Primavera (10.6x) vs. San Victorino (0.08x) |
| `razon_finde_habil` | Si la estacion vive del commuting entre semana o tiene vida propia el fin de semana | Dimension independiente de la anterior (correlacion ~0.1); separa estaciones de oficina de estaciones de ocio/comercio | Seccion 2.5: hábil vs. sabado/domingo tienen formas de curva muy distintas |
| `indice_concentracion_pico` | Que tan picuda es la curva de esa estacion (vs. una demanda pareja todo el dia) | Insumo directo para la pregunta especifica 5 (donde reforzar personal en horas puntuales vs. estaciones con flujo constante) | Seccion 2.7: rango de 2.07 a 24 entre estaciones -- hay mucha variacion real que capturar |
| Curva horaria normalizada completa (24 valores `h00`...`h23`, ya en la tabla de features) | La forma completa del dia, sin resumirla en un solo numero | Alternativa/complemento a las razones de arriba si el clustering con pocas variables no separa bien arquetipos (ej. una estacion con 2 picos marcados vs. una con una meseta ancha pueden tener la misma `razon_am_pm` y verse distinto en la curva completa) | Seccion 2.7 (construccion) |

**Lo que se deja fuera y por que:**
- `pct_pico_am`, `pct_pico_pm`: redundantes con `razon_am_pm` (ver matriz arriba).
- Variables de `Tipo_Tarifa`/`Nombre_Perfil` (perfil socioeconomico de las tarjetas): se exploraron las columnas (seccion 1.3) pero no se agregaron a nivel de estacion en este EDA -- queda como extension si el clustering inicial no separa bien y se necesita una dimension socioeconomica adicional.

### Conclusiones del EDA (antes de pasar a Dask y al modelo)

1. Hay **31.5 millones de validaciones reales** limpiadas y agregadas sin perdida de informacion.
2. La demanda de TransMilenio troncal tiene una **estacionalidad diaria fuerte y no estacionaria** (confirmado con Dickey-Fuller y descomposicion), con un patron claramente distinto entre dia habil y fin de semana.
3. La demanda esta **geograficamente concentrada** (Gini=0.484) y su centro de gravedad **se mueve por la ciudad segun la hora** (portales en la manana, zona comercial en la tarde, vida nocturna/eventos de noche).
4. Hay **arquetipos de estacion claramente distintos** incluso antes de correr un algoritmo formal (residencial vs. destino), lo que da confianza en que el clustering del notebook 04 va a encontrar grupos interpretables.
5. El festivo analizado **se comporta como un domingo**, un dato puntual util para la planeacion operativa, documentado por separado para no contaminar el resto del analisis.
6. Las variables candidatas para el modelo quedaron **revisadas por redundancia**, no solo listadas a ojo -- ver tabla arriba.